# 🧪 Practice Lab: Python for GenAI

**Netsetos GenAI Course** | Module 0 · Lesson 0.1 | ⏱ ~60-90 min

**Prerequisites:** Python 3.10+, `pip install pydantic`

### 🎯 You will:
1. Run async LLM calls 5-10× faster with `asyncio.gather()`
2. Simulate token streaming with generators
3. Validate LLM outputs with Pydantic models
4. Handle API errors with fallback chains
5. Stack decorators: @cache + @retry + @timer
6. Build a production LLM pipeline combining all skills

---

## Exercise 1 (Easy): Async Speedup — 10 Parallel Calls

**📖 Concept:** LLM API calls are I/O-bound — 95% waiting for network. `asyncio.gather()` fires all calls simultaneously. 10 calls × 2s each: sequential = 20s, async = 2s.

**📝 Task:**
1. Write `async def call_llm(prompt)` that simulates a 2-second API call with `await asyncio.sleep(2)`
2. Create 10 prompts. Measure sequential time (one-by-one) vs parallel time (`asyncio.gather()`)
3. Print the speedup ratio

**📤 Expected Output:**
```
Sequential: 20.0s
Parallel:   2.0s
Speedup:    10.0x
```

In [ ]:
# ✏️ YOUR CODE HERE
import asyncio, time

async def call_llm(prompt, delay=2.0):
    """Simulate an LLM API call."""
    # TODO: await the delay, return a response string
    pass

async def main():
    prompts = [f'Question {i}' for i in range(10)]

    # --- Sequential ---
    start = time.time()
    for p in prompts:
        pass  # TODO: await call_llm(p)
    seq_time = time.time() - start

    # --- Parallel ---
    start = time.time()
    # TODO: use asyncio.gather() to call all 10 in parallel
    par_time = time.time() - start

    print(f'Sequential: {seq_time:.1f}s')
    print(f'Parallel:   {par_time:.1f}s')
    print(f'Speedup:    {seq_time/par_time:.1f}x')

await main()  # Colab/Jupyter — use asyncio.run(main()) in scripts

<details><summary>💡 Hint 1 — Conceptual</summary>

`asyncio.gather()` runs all coroutines concurrently. Total time = slowest single call (2s), not the sum (20s).

</details>

<details><summary>💡 Hint 2 — Code</summary>

```python
results = await asyncio.gather(*[call_llm(p) for p in prompts])
```

</details>

**✅ Success Criteria:**
- call_llm() awaits asyncio.sleep and returns a response string
- Sequential loop takes ~20s
- Parallel gather takes ~2s
- Speedup ratio printed (~10x)

<details><summary>✅ Solution</summary>



In [ ]:
import asyncio, time

async def call_llm(prompt, delay=2.0):
    await asyncio.sleep(delay)
    return f'Response: {prompt}'

async def main():
    prompts = [f'Question {i}' for i in range(10)]

    # Sequential
    start = time.time()
    for p in prompts:
        await call_llm(p)
    seq_time = time.time() - start

    # Parallel
    start = time.time()
    results = await asyncio.gather(*[call_llm(p) for p in prompts])
    par_time = time.time() - start

    print(f'Sequential: {seq_time:.1f}s')
    print(f'Parallel:   {par_time:.1f}s')
    print(f'Speedup:    {seq_time/par_time:.1f}x')
    print(f'Results:    {len(results)} responses collected')

await main()

</details>

---

## Exercise 2 (Easy): Streaming Simulator

**📖 Concept:** Generators + `yield` display tokens as they arrive. TTFT drops from 2000ms to ~150ms. Every chat app (ChatGPT, Gemini, Claude) uses this pattern.

**📝 Task:**
1. Write `stream_chars(text, delay=0.05)` — yield one character at a time
2. Write `stream_words(text, delay=0.1)` — yield one word at a time
3. Print with `flush=True` to see the streaming effect

**📤 Expected Output:**
```
Character streaming:
Hello from GenAI!   (appears one letter at a time)

Word streaming:
RAG combines retrieval with generation   (appears one word at a time)
```

In [ ]:
# ✏️ YOUR CODE HERE
import time

def stream_chars(text, delay=0.05):
    """Yield one character at a time."""
    # TODO: loop through characters, sleep, yield each one
    pass

def stream_words(text, delay=0.1):
    """Yield one word at a time."""
    # TODO: split text into words, sleep, yield each word + space
    pass

# Test character streaming
print('Character streaming:')
for c in stream_chars('Hello from GenAI!'):
    print(c, end='', flush=True)
print()

# Test word streaming
print('\nWord streaming:')
for w in stream_words('RAG combines retrieval with generation for better answers'):
    print(w, end='', flush=True)
print()

<details><summary>💡 Hint 1 — Conceptual</summary>

`yield` pauses the function and returns one value. The caller gets values one at a time via `for...in`. The function resumes where it left off on the next iteration.

</details>

<details><summary>💡 Hint 2 — Code</summary>

```python
for char in text:
    time.sleep(delay)
    yield char
```

</details>

**✅ Success Criteria:**
- stream_chars yields individual characters with delay
- stream_words yields individual words with delay
- flush=True used for real-time display

<details><summary>✅ Solution</summary>



In [ ]:
import time

def stream_chars(text, delay=0.05):
    for c in text:
        time.sleep(delay)
        yield c

def stream_words(text, delay=0.1):
    for w in text.split():
        time.sleep(delay)
        yield w + ' '

print('Character streaming:')
for c in stream_chars('Hello from GenAI!'):
    print(c, end='', flush=True)
print()

print('\nWord streaming:')
for w in stream_words('RAG combines retrieval with generation for better answers'):
    print(w, end='', flush=True)
print()

</details>

---

## Exercise 3 (Medium): Pydantic Model Library

**📖 Concept:** Pydantic is the #1 Python library for GenAI. It validates types AND constraints. When an LLM returns `{"rating": "high"}` instead of `{"rating": 8}`, Pydantic catches it instantly.

**📝 Task:**
1. **ChatMessage** — role (user/assistant/system), content (1-10000 chars), optional tokens
2. **CourseRecommendation** — name, relevance_score (0-1), difficulty level, estimated_hours, topics
3. **CodeReview** — file_name, overall_score (1-10), issues list
4. Test with valid AND invalid data to show Pydantic catching bad LLM output

**📤 Expected Output:**
```
✅ Valid ChatMessage: assistant — Hello!
❌ Caught bad role: ValidationError
❌ Caught bad recommendation: ValidationError
```

In [ ]:
# ✏️ YOUR CODE HERE
from pydantic import BaseModel, Field
from typing import Optional

class ChatMessage(BaseModel):
    # TODO: role with pattern validation (user|assistant|system)
    # TODO: content with min_length=1, max_length=10000
    # TODO: optional tokens field (>= 0)
    pass

class CourseRecommendation(BaseModel):
    # TODO: course_name (min 3 chars)
    # TODO: relevance_score (0.0 to 1.0)
    # TODO: difficulty (beginner|intermediate|advanced)
    # TODO: estimated_hours (1 to 500)
    # TODO: topics (at least 1)
    pass

class CodeReview(BaseModel):
    # TODO: file_name, overall_score (1-10), issues list
    pass

# --- Test valid data ---
msg = ChatMessage(role='assistant', content='Hello!', tokens=5)
print(f'✅ Valid ChatMessage: {msg.role} — {msg.content}')

# --- Test invalid data ---
try:
    ChatMessage(role='hacker', content='test')
except Exception as e:
    print(f'❌ Caught bad role: {type(e).__name__}')

try:
    CourseRecommendation(
        course_name='AI', relevance_score=1.5,
        difficulty='expert', estimated_hours=0, topics=[]
    )
except Exception as e:
    print(f'❌ Caught bad recommendation: {type(e).__name__}')

<details><summary>💡 Hint 1 — Conceptual</summary>

`Field(..., ge=0, le=1)` enforces numeric ranges. `pattern='^(a|b|c)$'` enforces string enum values via regex.

</details>

<details><summary>💡 Hint 2 — Code</summary>

```python
role: str = Field(..., pattern='^(user|assistant|system)$')
content: str = Field(..., min_length=1, max_length=10000)
tokens: Optional[int] = Field(None, ge=0)
```

</details>

**✅ Success Criteria:**
- All 3 Pydantic models defined with proper Field constraints
- Valid data parses successfully
- Invalid role raises ValidationError
- Out-of-range score raises ValidationError

<details><summary>✅ Solution</summary>



In [ ]:
from pydantic import BaseModel, Field
from typing import Optional

class ChatMessage(BaseModel):
    role: str = Field(..., pattern='^(user|assistant|system)$')
    content: str = Field(..., min_length=1, max_length=10000)
    tokens: Optional[int] = Field(None, ge=0)

class CourseRecommendation(BaseModel):
    course_name: str = Field(..., min_length=3)
    relevance_score: float = Field(..., ge=0.0, le=1.0)
    difficulty: str = Field(..., pattern='^(beginner|intermediate|advanced)$')
    estimated_hours: int = Field(..., ge=1, le=500)
    topics: list[str] = Field(..., min_length=1)

class CodeReview(BaseModel):
    file_name: str
    overall_score: int = Field(..., ge=1, le=10)
    issues: list[str] = Field(default_factory=list)

# --- Test valid data ---
msg = ChatMessage(role='assistant', content='Hello!', tokens=5)
print(f'✅ Valid ChatMessage: {msg.role} — {msg.content}')

course = CourseRecommendation(
    course_name='GenAI Engineering',
    relevance_score=0.95,
    difficulty='advanced',
    estimated_hours=100,
    topics=['LLM', 'RAG', 'agents']
)
print(f'✅ Valid CourseRecommendation: {course.course_name} ({course.relevance_score})')

review = CodeReview(file_name='app.py', overall_score=8, issues=['missing docstrings'])
print(f'✅ Valid CodeReview: {review.file_name} ({review.overall_score}/10)')

# --- Test invalid data ---
try:
    ChatMessage(role='hacker', content='test')
except Exception as e:
    print(f'❌ Caught bad role: {type(e).__name__}')

try:
    CourseRecommendation(
        course_name='AI', relevance_score=1.5,
        difficulty='expert', estimated_hours=0, topics=[]
    )
except Exception as e:
    print(f'❌ Caught bad recommendation: {type(e).__name__}')

</details>

---

## Exercise 4 (Medium): Error Handling with Fallback Chain

**📖 Concept:** LLM APIs fail: 429 (rate limit) → wait and retry. 500 (server error) → retry. 401 (auth) → FAIL FAST. The production pattern: **exponential backoff with jitter** + **model fallback chain** (primary → cheaper → cached).

**📝 Task:**
1. Create 3 custom exception classes: `RateLimitError`, `ServerError`, `AuthError`
2. Write `retry_with_backoff(func, prompt, retries=3)` — exponential backoff + jitter, fail fast on AuthError
3. Write `call_with_fallback(prompt)` — tries 3 models in sequence: primary → cheaper → cached
4. Test with a 40% failure rate simulated API

**📤 Expected Output:**
```
  Trying primary...
  Rate limited, waiting 1.6s
  Result: Answer: GenAI concept 0

  Trying primary...
  Result: Answer: GenAI concept 1
```

In [ ]:
# ✏️ YOUR CODE HERE
import time, random

class RateLimitError(Exception): pass
class ServerError(Exception): pass
class AuthError(Exception): pass

def call_unreliable(prompt):
    """Simulate an unreliable LLM API (40% failure rate)."""
    r = random.random()
    if r < 0.3: raise RateLimitError('429: Too Many Requests')
    if r < 0.4: raise ServerError('500: Internal Server Error')
    return f'Answer: {prompt}'

def retry_with_backoff(func, prompt, retries=3):
    """Retry with exponential backoff + jitter. Fail fast on AuthError."""
    for attempt in range(retries):
        try:
            return func(prompt)
        except AuthError:
            # TODO: print auth failure, re-raise immediately (don't retry)
            pass
        except RateLimitError:
            # TODO: calculate wait = 2^attempt + random jitter
            # TODO: print wait time, sleep (use * 0.01 for fast testing)
            pass
        except ServerError:
            # TODO: print retry message, short sleep
            pass
    raise Exception(f'All {retries} retries failed')

def call_with_fallback(prompt):
    """Try models in order: primary → cheaper → cached."""
    # TODO: loop through ['primary', 'cheaper', 'cached']
    # TODO: for 'cached', return a cached response directly
    # TODO: for others, call retry_with_backoff(call_unreliable, prompt)
    pass

# Test
random.seed(42)
for i in range(5):
    result = call_with_fallback(f'GenAI concept {i}')
    print(f'  Result: {result}\n')

<details><summary>💡 Hint 1 — Conceptual</summary>

Exponential backoff: delay = 2^attempt seconds. Jitter adds randomness (`random.uniform(0, 1)`) to prevent thundering herd. AuthError should never retry — the key is wrong, retrying won't fix it.

</details>

<details><summary>💡 Hint 2 — Code</summary>

```python
# In retry_with_backoff:
except RateLimitError:
    wait = (2 ** attempt) + random.uniform(0, 1)
    print(f'  Rate limited, waiting {wait:.1f}s')
    time.sleep(wait * 0.01)  # scaled down for testing
```

</details>

**✅ Success Criteria:**
- 3 custom exception classes defined
- retry_with_backoff implements exponential backoff with jitter
- AuthError causes immediate failure (no retry)
- call_with_fallback tries models in order, cached always succeeds

<details><summary>✅ Solution</summary>



In [ ]:
import time, random

class RateLimitError(Exception): pass
class ServerError(Exception): pass
class AuthError(Exception): pass

def call_unreliable(prompt):
    """Simulate an unreliable LLM API (40% failure rate)."""
    r = random.random()
    if r < 0.3:
        raise RateLimitError('429: Too Many Requests')
    if r < 0.4:
        raise ServerError('500: Internal Server Error')
    return f'Answer: {prompt}'

def retry_with_backoff(func, prompt, retries=3):
    """Retry with exponential backoff + jitter. Fail fast on AuthError."""
    for attempt in range(retries):
        try:
            return func(prompt)
        except AuthError:
            print('  Auth fail — STOP (not retrying)')
            raise
        except RateLimitError:
            wait = (2 ** attempt) + random.uniform(0, 1)
            print(f'  Rate limited, waiting {wait:.1f}s')
            time.sleep(wait * 0.01)  # Scaled down for testing
        except ServerError:
            print(f'  Server error, retry {attempt + 1}')
            time.sleep(0.05)
    raise Exception(f'All {retries} retries failed')

def call_with_fallback(prompt):
    """Try models in order: primary → cheaper → cached."""
    for model in ['primary', 'cheaper', 'cached']:
        try:
            print(f'  Trying {model}...')
            if model == 'cached':
                return f'[Cached] {prompt}'
            return retry_with_backoff(call_unreliable, prompt)
        except Exception as e:
            print(f'  {model} failed: {e}')
    return 'Unavailable'

# Test
random.seed(42)
for i in range(5):
    result = call_with_fallback(f'GenAI concept {i}')
    print(f'  Result: {result}\n')

</details>

---

## Exercise 5 (Hard): Decorator Stack — @cache + @retry + @timer

**📖 Concept:** Production LLM calls stack decorators: `@timer @retry @cache def func`. Order matters — cache is checked FIRST (innermost), retry wraps it, timer measures total. A cached call saves ~₹0.08. At 100K calls/day, caching 40% = ₹3,200/day saved.

**📝 Task:**
1. Write `@cache_response` — dict-based cache, prints CACHE HIT on repeat calls
2. Write `@retry(max_attempts=3, delay=0.5)` — exponential backoff on failure
3. Write `@timer` — prints execution time in milliseconds
4. Stack all three on a `call_llm()` function with 30% simulated failure rate
5. Call twice with same prompt — second call should be instant (cached)

**📤 Expected Output:**
```
Call 1 (new):
  512ms
Answer: What is RAG?

Call 2 (should be cached):
  CACHE HIT
  0ms
Answer: What is RAG?
```

In [ ]:
# ✏️ YOUR CODE HERE
import time, functools, random

def cache_response(func):
    """Cache results by arguments. Print 'CACHE HIT' on repeat."""
    _cache = {}
    @functools.wraps(func)
    def wrapper(*args):
        key = str(args)
        # TODO: check if key in cache, return cached if so
        # TODO: otherwise call func, store result, return it
        pass
    return wrapper

def retry(max_attempts=3, delay=0.5):
    """Retry with exponential backoff."""
    def decorator(func):
        @functools.wraps(func)
        def wrapper(*args):
            for attempt in range(max_attempts):
                try:
                    return func(*args)
                except Exception:
                    if attempt < max_attempts - 1:
                        time.sleep(delay * (2 ** attempt))
                    else:
                        raise
        return wrapper
    return decorator

def timer(func):
    """Print execution time in milliseconds."""
    @functools.wraps(func)
    def wrapper(*args):
        # TODO: measure time, call func, print ms, return result
        pass
    return wrapper

@timer
@retry(max_attempts=3, delay=0.3)
@cache_response
def call_llm(prompt):
    if random.random() < 0.3:
        raise ConnectionError('API timeout')
    time.sleep(0.5)  # Simulate API latency
    return f'Answer: {prompt}'

# Test
print('Call 1 (new):')
print(call_llm('What is RAG?'))
print('\nCall 2 (should be cached):')
print(call_llm('What is RAG?'))

<details><summary>💡 Hint 1 — Conceptual</summary>

Decorator execution order: `@timer @retry @cache def f` means `timer(retry(cache(f)))`. When you call f(), timer runs first (measuring), then retry wraps cache(f), and cache checks the dict before calling the real function.

</details>

<details><summary>💡 Hint 2 — Code</summary>

```python
# cache_response:
if key in _cache:
    print('  CACHE HIT')
    return _cache[key]
result = func(*args)
_cache[key] = result
return result
```

</details>

**✅ Success Criteria:**
- @cache_response stores and retrieves results by argument key
- @retry retries with exponential backoff, re-raises after max_attempts
- @timer prints execution time in milliseconds
- Second call with same prompt hits cache (instant, ~0ms)
- All three decorators stack correctly

<details><summary>✅ Solution</summary>



In [ ]:
import time, functools, random

def cache_response(func):
    """Cache results by arguments."""
    _cache = {}
    @functools.wraps(func)
    def wrapper(*args):
        key = str(args)
        if key in _cache:
            print('  CACHE HIT')
            return _cache[key]
        result = func(*args)
        _cache[key] = result
        return result
    return wrapper

def retry(max_attempts=3, delay=0.5):
    """Retry with exponential backoff."""
    def decorator(func):
        @functools.wraps(func)
        def wrapper(*args):
            for attempt in range(max_attempts):
                try:
                    return func(*args)
                except Exception:
                    if attempt < max_attempts - 1:
                        wait = delay * (2 ** attempt)
                        time.sleep(wait)
                    else:
                        raise
        return wrapper
    return decorator

def timer(func):
    """Print execution time in milliseconds."""
    @functools.wraps(func)
    def wrapper(*args):
        start = time.time()
        result = func(*args)
        elapsed_ms = (time.time() - start) * 1000
        print(f'  {elapsed_ms:.0f}ms')
        return result
    return wrapper

@timer
@retry(max_attempts=3, delay=0.3)
@cache_response
def call_llm(prompt):
    if random.random() < 0.3:
        raise ConnectionError('API timeout')
    time.sleep(0.5)
    return f'Answer: {prompt}'

print('Call 1 (new):')
print(call_llm('What is RAG?'))
print('\nCall 2 (cached):')
print(call_llm('What is RAG?'))

</details>

---

## Exercise 6 (Hard): Production LLM Pipeline — ALL Skills Combined

**📖 Concept:** This combines ALL 7 skills into the exact architecture used in Modules 5–11: Pydantic schema → async parallel calls → retry on failure → cache duplicates → validate response → stream output.

**📝 Task:**
1. Define Pydantic model: `LLMResponse(answer, confidence, tokens)`
2. Write an async `@cache` decorator
3. Write async `call_llm(prompt)` with retry + Pydantic validation
4. Write a generator `stream(text)` for word-by-word output
5. Run 4 prompts (including 1 duplicate) in parallel with `asyncio.gather()`
6. Stream each result word-by-word

**📤 Expected Output:**
```
  CACHE: RAG
4 calls in 0.8s

[92%, 87tok] GenAI explanation for RAG
[85%, 142tok] GenAI explanation for agents
[78%, 55tok] GenAI explanation for MCP
[92%, 87tok] GenAI explanation for RAG
```

In [ ]:
# ✏️ YOUR CODE HERE
import asyncio, time, functools, random
from pydantic import BaseModel, Field

# --- Step 1: Pydantic model ---
class LLMResponse(BaseModel):
    answer: str = Field(..., min_length=1)
    confidence: float = Field(..., ge=0, le=1)
    tokens: int = Field(..., ge=1)

# --- Step 2: Async cache decorator ---
def cache(func):
    _cache = {}
    @functools.wraps(func)
    async def wrapper(*args):
        key = str(args)
        # TODO: check cache, return cached if hit
        # TODO: otherwise await func, store, return
        pass
    return wrapper

# --- Step 3: Async LLM call with retry + validation ---
@cache
async def call_llm(prompt):
    for attempt in range(3):
        try:
            await asyncio.sleep(random.uniform(0.3, 0.8))
            if random.random() < 0.2:
                raise ConnectionError('timeout')
            # TODO: return LLMResponse with simulated data
            pass
        except ConnectionError:
            if attempt < 2:
                await asyncio.sleep(2 ** attempt * 0.1)
            else:
                raise

# --- Step 4: Streaming generator ---
def stream(text, delay=0.02):
    # TODO: yield words one at a time with delay
    pass

# --- Step 5: Main pipeline ---
async def main():
    prompts = ['RAG', 'agents', 'MCP', 'RAG']  # RAG appears twice (cache test)
    start = time.time()
    # TODO: gather all calls in parallel
    # TODO: print timing
    # TODO: stream each result word-by-word

await main()

<details><summary>💡 Hint 1 — Conceptual</summary>

The async cache wrapper must use `async def` and `await func(*args)`. The 4th prompt ("RAG") should hit the cache from the 1st call and print "CACHE" instead of making a new API call.

</details>

<details><summary>💡 Hint 2 — Code</summary>

```python
# Async cache:
if key in _cache:
    print(f'  CACHE: {args[0]}')
    return _cache[key]
result = await func(*args)
_cache[key] = result
return result

# LLMResponse:
return LLMResponse(
    answer=f'GenAI explanation for {prompt}',
    confidence=round(random.uniform(0.7, 0.99), 2),
    tokens=random.randint(30, 150)
)
```

</details>

**✅ Success Criteria:**
- Pydantic model validates all LLM responses
- Async cache prevents duplicate API calls
- Retry handles transient failures with backoff
- 4 prompts run in parallel (~0.8s, not ~3.2s)
- Duplicate "RAG" prompt hits cache
- Results streamed word-by-word

<details><summary>✅ Solution</summary>



In [ ]:
import asyncio, time, functools, random
from pydantic import BaseModel, Field

# --- Pydantic model ---
class LLMResponse(BaseModel):
    answer: str = Field(..., min_length=1)
    confidence: float = Field(..., ge=0, le=1)
    tokens: int = Field(..., ge=1)

# --- Async cache decorator ---
def cache(func):
    _cache = {}
    @functools.wraps(func)
    async def wrapper(*args):
        key = str(args)
        if key in _cache:
            print(f'  CACHE: {args[0]}')
            return _cache[key]
        result = await func(*args)
        _cache[key] = result
        return result
    wrapper.cache = _cache
    return wrapper

# --- Async LLM call with retry + validation ---
@cache
async def call_llm(prompt):
    for attempt in range(3):
        try:
            await asyncio.sleep(random.uniform(0.3, 0.8))
            if random.random() < 0.2:
                raise ConnectionError('timeout')
            return LLMResponse(
                answer=f'GenAI explanation for {prompt}',
                confidence=round(random.uniform(0.7, 0.99), 2),
                tokens=random.randint(30, 150)
            )
        except ConnectionError:
            if attempt < 2:
                await asyncio.sleep(2 ** attempt * 0.1)
            else:
                raise

# --- Streaming generator ---
def stream(text, delay=0.02):
    for word in text.split():
        time.sleep(delay)
        yield word + ' '

# --- Main pipeline ---
async def main():
    prompts = ['RAG', 'agents', 'MCP', 'RAG']  # RAG is duplicate (cache test)
    start = time.time()

    results = await asyncio.gather(*[call_llm(p) for p in prompts])
    print(f'4 calls in {time.time() - start:.1f}s\n')

    for r in results:
        print(f'[{r.confidence:.0%}, {r.tokens}tok] ', end='')
        for w in stream(r.answer):
            print(w, end='', flush=True)
        print()

await main()

</details>

---

## 🎉 Done!

You've built the complete Python toolkit for GenAI:
- **Async/Await** — parallel API calls
- **Generators** — token streaming
- **Type Hints + Pydantic** — LLM output validation
- **Decorators** — retry, cache, timer
- **Error Handling** — rate limits, fallback chains
- **Production Pattern** — all skills combined

No other GenAI course teaches this integrated set. Next: **Lesson 0.2 Math for GenAI.**